# Experiment 012 (residual information after the 009 scalar-LEACE erasure) on Google Colab
Order: **A** settings + setup (A1 GPU, A2 settings, A3 clone, A4 environment, A5 ACWM source, A6 env vars, A6b metadata.pt, A7 disk), **B** check (inputs, dry runs), **C** selftest (CPU toy), **D** optional pilot (1 train seed, block 6), **E** main `measure` (gate `APPROVED_MAIN_012`), **F** aggregate, **G** copy results out.
Every cell is idempotent; after a disconnect choose a GPU runtime, run A and B again and continue: `measure` skips finished units (part files on Drive).

What is measured (config `experiments/012_erasure_residual_info/config.yaml`, user-confirmed 2026-11-10, **not changed by this notebook**): at the 009 erasure (`leace_scalar`, strength 1.0, fit episodes `fit_seed` 9000) and at the strength-0 baseline of the same depth, for depth `m_int` (verdict) and `m_int+1` (auxiliary): (a) 3-level logistic accuracy, (b) MLP and attentive R2, (c) R2 from the action/proprio coordinates only, (d) linear R2 (reference, compared with 009). Probes are trained only on train episodes NOT used for the direction fit and evaluated on test episodes. Planned units: 21 = 3 baselines + 6 blocks x 3 seeds.
The only overrides used anywhere are existing CLI flags of `run.py` (pilot cell: `--m-int 6 --train-seeds 0 --results-dir`), which `results.json` records and marks `pilot: true`.

Inputs (read-only): the three 002 condition-(b) checkpoints `exp002_work/train_runs/outputs/exp002_multi_60_ind_160_s{0,1,2}/checkpoints/model_latest.pth` (note the `exp002_` prefix; 006 failed 3 times without it), the 002 Q1 cache, and the per-split `metadata.pt` files (cell A6b; 009's first pilot failed without them). **No world model is trained.** `run.py measure` has a preflight (exit code 4, no attempt consumed) for checkpoints, cache and `metadata.pt`.
Outputs (Drive): `MyDrive/exp012_work/{results,logs,results_pilot,timing_*.json,code_version.json}`; `results.json` is written to `exp012_work/results/` by aggregate.

Human approval (CLAUDE.md): GPU runs need you. The pilot needs `APPROVED_PILOT_012 = True`, the main run needs `APPROVED_MAIN_012 = True` (cell D0). All time numbers printed from docs are estimates (추정). Everything here is UNVERIFIED on Colab (written and checked on a CPU-only machine).

In [ ]:
# A1. GPU / Python / disk
import sys, subprocess, os
def sh(cmd, cwd=None, env=None):
    env = dict(os.environ if env is None else env, MPLBACKEND='Agg')
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    return p.wait()
sh('nvidia-smi')
print(sys.version)
sh('df -h /content | tail -1')
sh('free -g | head -2')

In [ ]:
# A2. SETTINGS (all Drive paths and options live here). Mount Drive first, then check that the paths below are right for YOUR Drive.
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK002 = os.environ.get('EXP002_WORK_ROOT', DRIVE + '/exp002_work')     # READ-ONLY: train_runs/outputs/exp002_multi_60_ind_160_s{0,1,2}/ (checkpoints), exp002_cache/q1_tokens/, results/results.json
WORK012 = os.environ.get('EXP012_WORK_ROOT', DRIVE + '/exp012_work')     # OUTPUT (persistent): results/, logs/, results_pilot/, timing_*.json
DATA_SRC = os.environ.get('EXP002_DRIVE_MOUNT_SRC', DRIVE + '/dino-wm-videos/[data]/[cloth-sim] cloth_move')   # same as the 004/009 notebooks; only <split>/metadata.pt is read from here (cell A6b)
REPO_URL = 'https://github.com/jongmin-s-multicore/probing-experiment.git'     # PUBLIC repository (cloned in A3)
REPO_REF = 'master'            # branch name OR commit hash (or tag) to check out in A3; use a commit hash for an exactly reproducible run
GITHUB_TOKEN = None            # optional; not needed for the public repo (e.g. userdata.get('GITHUB_TOKEN') if you ever use a private fork)
LOCAL_ROOT = '/content/cache012'                                         # big temporary features (up to ~13 GB for a baseline unit, 추정) on the Colab local disk
os.makedirs(WORK012, exist_ok=True)
for k, v in (('WORK002', WORK002), ('WORK012', WORK012)):
    print('%-9s = %s  exists=%s' % (k, v, os.path.exists(v)))
for ts in (0, 1, 2):
    p = '%s/train_runs/outputs/exp002_multi_60_ind_160_s%d/checkpoints/model_latest.pth' % (WORK002, ts)
    print('checkpoint seed %d: %s exists=%s' % (ts, p, os.path.exists(p)))

In [ ]:
# A3. Code: git clone of the PUBLIC repo REPO_URL, checked out at REPO_REF (settings in cell A2). No embedded snapshot.
# The repo ref must contain 002, 004 (colab/exp045_tools.py etc.), 009 (src, config.yaml, results.json) and 012 files (commit and push them first).
EXPECTED_CFG002_SHA = '4d5261d68a5d7270d845bab6ef30f924bad31fd46f0b252b802303a3823fd1ce'    # 002/results.json config_sha256 read at notebook build time (2026-11-10 20:53:26)
import json, os, subprocess
REPO = '/content/mylab012'
EXPS = REPO + '/experiments'
EXP002, EXP004, EXP009, EXP012 = EXPS + '/002_single_size_sweep', EXPS + '/004_depth_truncation', EXPS + '/009_positive_control_depth', EXPS + '/012_erasure_residual_info'
def git(*a, check=True):
    r = subprocess.run(['git', '-C', REPO] + list(a), capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError('git %s failed: %s' % (' '.join(a[:2]), r.stderr.replace(GITHUB_TOKEN or '@@none@@', '***')))
    return r.stdout.strip()
clone_url = REPO_URL
if GITHUB_TOKEN:        # optional (public repo needs none)
    clone_url = REPO_URL.replace('https://', 'https://%s@' % GITHUB_TOKEN)
if not os.path.isdir(REPO + '/.git'):
    r = subprocess.run(['git', 'clone', clone_url, REPO], capture_output=True, text=True)
    assert r.returncode == 0, 'clone failed: ' + r.stderr.replace(GITHUB_TOKEN or '@@none@@', '***')
else:
    git('remote', 'set-url', 'origin', clone_url)
    git('fetch', '--tags', '--force', 'origin', '+refs/heads/*:refs/remotes/origin/*')
git('remote', 'set-url', 'origin', REPO_URL)          # do not keep the token in .git/config
if subprocess.run(['git', '-C', REPO, 'rev-parse', '--verify', '-q', 'origin/' + REPO_REF + '^{commit}'], capture_output=True).returncode == 0:
    target = 'origin/' + REPO_REF                       # branch name
else:
    target = REPO_REF                                   # commit hash or tag
git('checkout', '--force', '--detach', target)
HEAD_SHA = git('rev-parse', 'HEAD')
CODE_VERSION = {'repo_url': REPO_URL, 'requested_ref': REPO_REF, 'git_head': HEAD_SHA, 'head_subject': git('log', '-1', '--format=%s'), 'head_date': git('log', '-1', '--format=%cI')}
print('code at HEAD', HEAD_SHA, '(ref %s)' % REPO_REF)
print(git('log', '--oneline', '-1'))
os.makedirs(WORK012, exist_ok=True)
json.dump(CODE_VERSION, open(WORK012 + '/code_version.json', 'w'), indent=1)     # also in results.json (code_version, via env vars set in A6)
for p in (EXP002 + '/run.py', EXP004 + '/run.py', EXP012 + '/run.py', EXP012 + '/config.yaml', EXP012 + '/src/exp012/stages.py', EXP009 + '/src/exp009/erase.py', EXP009 + '/config.yaml', EXP009 + '/results.json',
          EXP004 + '/colab/exp045_tools.py', EXP002 + '/colab/requirements_colab.txt', EXP004 + '/colab/requirements_colab_extra.txt'):
    assert os.path.isfile(p), 'missing in this ref: ' + p + ' (commit and push the code, or fix REPO_REF)'
import sys
sys.path.insert(0, EXP004 + '/colab')
import importlib, exp045_tools as T
importlib.reload(T)
print('exp045_tools loaded from', T.__file__)

In [ ]:
# A4. Python environment: uv venv (3.9) with the 002 pins (+ matplotlib). Colab has no conda: documented exception to CLAUDE.md (same as 002/003/004/009).
# USE_SYSTEM_PYTHON=True uses Colab's own Python/torch; results are then NOT comparable with the 002 environment (record it). torch 2.0.x comes from the 002 pins.
USE_SYSTEM_PYTHON = False
VENV = '/content/venv012'
import hashlib, os, sys
REQ = [EXP002 + '/colab/requirements_colab.txt', EXP004 + '/colab/requirements_colab_extra.txt']
if USE_SYSTEM_PYTHON:
    PY = sys.executable
else:
    PY = VENV + '/bin/python'
    MARK = VENV + '/.exp012_ok_' + hashlib.sha256(b''.join(open(r, 'rb').read() for r in REQ)).hexdigest()[:12]
    if not os.path.exists(MARK):
        assert sh('pip install -q uv') == 0
        if not os.path.exists(PY):
            assert sh('uv venv --python 3.9 %s' % VENV) == 0
        assert sh('uv pip install --python %s --index-strategy unsafe-best-match -r %s -r %s' % (PY, REQ[0], REQ[1])) == 0
        open(MARK, 'w').write('ok\n')
CHK = "import sys,pkg_resources,torch,numpy,cv2,decord,hydra,yaml;print(sys.version.split()[0],'torch',torch.__version__,'cuda',torch.version.cuda,torch.cuda.is_available(),'numpy',numpy.__version__)"
assert sh('%s -c "%s"' % (PY, CHK)) == 0, 'environment check failed'

In [ ]:
# A5. ACWM-Phys checkout at the pinned commit (== 002 config.yaml code.commit). Needed to unpickle the 002 checkpoints. NO resume patch (nothing is trained).
import re, subprocess, os
cfg002_txt = open(EXP002 + '/config.yaml').read()
COMMIT = re.search(r'commit:\s*"([0-9a-f]{40})"', cfg002_txt).group(1)
SRC = '/content/dinowm/acwm_phys_src'
if not os.path.isdir(SRC + '/.git'):
    os.makedirs(os.path.dirname(SRC), exist_ok=True)
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/jongmin-s-multicore/thread0.git', SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'checkout', '--detach', COMMIT], check=True)
head = subprocess.run(['git', '-C', SRC, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, (head, COMMIT)
print('ACWM source at', head)

In [ ]:
# A6. Environment variables, Drive-backed results/logs, stage helpers.
import os, json, shutil
for d in (LOCAL_ROOT + '/exp012_cache', '/content/pilot012/exp012_cache'):
    os.makedirs(d, exist_ok=True)
base_env = dict(os.environ)
base_env.update({
    'DINO_WORK': '/content/dinowm', 'DATASET_DIR': '/content/dinowm/data',
    'DINO_TRAIN': WORK002 + '/train_runs',                                          # READ-ONLY use: checkpoints (<DINO_TRAIN>/outputs/exp002_multi_60_ind_160_s<seed>/)
    'EXP002_CACHE': WORK002 + '/exp002_cache',                                      # READ-ONLY use: q1_tokens
    'ACWM_PHYS_SRC': SRC, 'WANDB_MODE': 'disabled', 'HYDRA_FULL_ERROR': '1', 'PYTHONUNBUFFERED': '1',
    'EXP012_CODE_REPO_URL': REPO_URL, 'EXP012_CODE_REF': REPO_REF,                  # recorded in results.json code_version (git_head is read from the clone)
    'MPLBACKEND': 'Agg',                                                            # also forced inside exp045_tools (the Colab kernel's inline backend is invalid in the venv)
    'PATH': os.path.dirname(PY) + os.pathsep + os.environ['PATH'],
})
# NOTE: PYTORCH_CUDA_ALLOC_CONF is deliberately not set (expandable_segments is unknown to torch 2.0.x; 003 commit 16edd98).
if not USE_SYSTEM_PYTHON:
    base_env['VIRTUAL_ENV'] = VENV
ENV_MAIN = dict(base_env, EXP012_CACHE=LOCAL_ROOT + '/exp012_cache')
ENV_PILOT = dict(base_env, EXP012_CACHE='/content/pilot012/exp012_cache')
print(json.dumps({k: ENV_MAIN[k] for k in ('DINO_TRAIN', 'EXP002_CACHE', 'EXP012_CACHE', 'ACWM_PHYS_SRC', 'MPLBACKEND')}, indent=1))
print('012:', T.link_dirs(WORK012, EXP012, ('results', 'logs')))
os.makedirs(WORK012 + '/results_pilot', exist_ok=True)
print('009 results.json (reference for the (d) reproduction check) sha256', T.sha256_file(EXP009 + '/results.json')[:16])

MODES = {      # pilot and main NEVER share a results folder: results.json of the pilot is written to results_pilot only and says pilot: true
    'pilot': dict(env=ENV_PILOT, results=WORK012 + '/results_pilot', timing=WORK012 + '/timing_pilot.json'),
    'main': dict(env=ENV_MAIN, results=WORK012 + '/results', timing=WORK012 + '/timing_main.json'),
}

def run012(stage, extra=(), mode='main', dry=False):
    m = MODES[mode]
    args = ['--results-dir', m['results']] + list(extra) + (['--dry-run'] if dry else [])
    rec = T.run_stage(PY, EXP012, stage, args, m['env'], timing_file=None if dry else m['timing'], label='012:%s%s' % (stage, '(dry)' if dry else ''))
    if rec['rc'] == 4 and not dry:
        raise SystemExit('PREFLIGHT FAILED (exit code 4, NO attempt consumed). Nothing was computed. Read the "PREFLIGHT ERROR" line above / log %s, fix the input (checkpoint folder, Q1 cache, metadata.pt, config), then run this cell again.' % rec['log'])
    T.require_ok(rec, m['results'])
    return rec

def save_results_json(mode):
    # aggregate writes results.json into the results folder (Drive); for the main run also <clone>/experiments/012.../results.json (lost with the session)
    src = (EXP012 + '/results.json') if mode == 'main' else (MODES[mode]['results'] + '/results.json')
    dst = MODES[mode]['results'] + '/results.json'
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copyfile(src, dst)
    print(('results.json present: ' + dst) if os.path.exists(dst) else ('no results.json at ' + dst))

In [ ]:
# A6b. metadata.pt of the training manifests (needed by exp002 model_io.action_stats_for_run -> ManifestDataset: action mean/std). Only metadata.pt files, no mp4.
# Same cell as 009 A6b. Sources in order: (a) already local; (b) DATA_SRC/<split>/metadata.pt or DATA_SRC/cloth_sim/<split>/metadata.pt; (c) member of WORK002/cloth_sim_data.tar (unsafe members refused).
import json, os, shutil, tarfile
from pathlib import Path
need = {}                                   # local dir (as written in the manifest) -> train seeds
for ts in (0, 1, 2):
    mp = Path(WORK002) / 'train_runs' / 'outputs' / ('exp002_multi_60_ind_160_s%d' % ts) / 'train_manifest.json'
    assert mp.exists(), 'missing ' + str(mp)
    for ep in json.load(open(mp))['episodes']:
        need.setdefault(ep['dir'], set()).add(ts)
print('episode dirs in the manifests:', {d: sorted(v) for d, v in need.items()})
def safe_member(name):
    return not name.startswith('/') and '..' not in name.split('/')
missing = []
tar_path = Path(WORK002) / 'cloth_sim_data.tar'
tf = None
for d in sorted(need):
    dest = Path(d) / 'metadata.pt'
    split = Path(d).name
    if dest.exists():
        print('local ok   ', dest); continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    got = None
    for cand in (Path(DATA_SRC) / split / 'metadata.pt', Path(DATA_SRC) / 'cloth_sim' / split / 'metadata.pt'):
        if cand.exists():
            shutil.copyfile(cand, dest); got = 'copied from ' + str(cand); break
    if got is None and tar_path.exists():
        if tf is None:
            tf = tarfile.open(tar_path, 'r:*')
        for m in tf:
            if not safe_member(m.name) or not m.isfile():
                continue
            parts = m.name.split('/')
            if parts[-1] == 'metadata.pt' and len(parts) >= 2 and parts[-2] == split:
                with tf.extractfile(m) as fi, open(dest, 'wb') as fo:
                    shutil.copyfileobj(fi, fo)
                got = 'extracted %s from %s' % (m.name, tar_path); break
    if got is None:
        missing.append(d); print('!! NOT FOUND', dest)
    else:
        print(got, '->', dest)
if tf is not None:
    tf.close()
if missing:
    raise SystemExit('metadata.pt not found for splits %s. Fix DATA_SRC in cell A2 or put cloth_sim_data.tar into %s. Nothing was run.' % ([Path(d).name for d in missing], WORK002))
print('metadata.pt present for all %d episode dirs' % len(need))

In [ ]:
# A7. Disk. Estimate (추정): a baseline unit writes the token features of 6 depths (about 2.2 GB each = 13 GB), an erased unit 2 depths (4.3 GB); deleted at the end of each unit.
T.disk_report(['/content', WORK012], need_gb={'012 tmp features (1 unit at a time, baseline unit largest)': 13})
free_local = shutil.disk_usage('/content').free / 1e9
print('free local disk: %.0f GB' % free_local, ('  !! LOW (< 25 GB)' if free_local < 25 else ''))

# B. Check (read-only; nothing is trained)

In [ ]:
# B1. Inputs on Drive (002 checkpoints DONE, final checkpoint, Q1 cache, 002 config fingerprint). Stops with a clear message if something is missing. Writes nothing.
ok, bad, lines = T.check_inputs(WORK002, [0, 1, 2], suffixes=('',), expected_epoch=15, expected_cfg_sha=EXPECTED_CFG002_SHA, cfg002_path=EXP002 + '/config.yaml', need_manifest=False)
print('\n'.join(lines))
if not ok:
    raise SystemExit('INPUTS MISSING - stopping (nothing was run):\n  - ' + '\n  - '.join(bad) + '\n  Fix the paths in cell A2 (WORK002).')
print('inputs OK')

In [ ]:
# B2. 012 `check` (config consistency with 009/002, checkpoint folder structure, Q1 cache, metadata.pt) and the dry runs (they print the plan: expected "units total 21 (baseline 3 + erased 18)"). No GPU work, no attempts recorded.
run012('check', mode='main')
run012('measure', mode='main', dry=True)
run012('aggregate', mode='main', dry=True)
print(T.read_yaml_values(EXP012 + '/config.yaml', ['pilot', 'seed', 'intervention.method', 'intervention.strengths', 'intervention.m_int', 'predictor.train_seeds', 'measure.probe_seeds', 'intervention.fit_seed', 'success']))

# C. Selftest (CPU, synthetic data + toy model; expected: failed=0)

In [ ]:
# C1. selftest (does not touch Drive inputs; it starts one subprocess that must exit with code 4 on purpose).
rec = run012('selftest', mode='main')
print('selftest %.0f s, log %s' % (rec['seconds'], rec['log']))

# D. OPTIONAL pilot: baseline (depth 6) + 1 erased unit (train seed 0, block 6, strength 1.0) with the REAL checkpoint, results in `exp012_work/results_pilot`
Uses only existing CLI flags (`--m-int 6 --train-seeds 0 --results-dir`; the pilot results.json says `pilot: true` and lists the reasons). Block 6 is the cheapest block (only depth 6 is forwarded, measured with all five measures) and has no m_int+1 observation; it measures time, VRAM and finds errors. Pilot and main never share a folder or cache.

In [ ]:
# D0. APPROVALS (CLAUDE.md: GPU/cost-heavy runs need human approval). Set by the human only.
APPROVED_PILOT_012 = False
APPROVED_MAIN_012 = False
ALLOW_MAIN_WITH_PILOT_TRUE = False      # True = run the main although config.yaml has pilot: true (results.json would then say pilot: true with that reason). Normally the human sets pilot: false in config.yaml first.
PILOT_M_INT = '6'                       # CLI --m-int for the pilot only
PILOT_TRAIN_SEEDS = '0'                 # CLI --train-seeds for the pilot only
print('APPROVED_PILOT_012 =', APPROVED_PILOT_012, '| APPROVED_MAIN_012 =', APPROVED_MAIN_012)

In [ ]:
# D1. Pilot: measure (baseline depth 6 for seed 0 + block 6 at strength 1.0) -> aggregate. Time and sampled VRAM go to exp012_work/timing_pilot.json.
import time
if not APPROVED_PILOT_012:
    print('사람 승인 필요(CLAUDE.md): APPROVED_PILOT_012 = False -> nothing was run.')
else:
    t0 = time.time()
    ex = ['--m-int', PILOT_M_INT, '--train-seeds', PILOT_TRAIN_SEEDS]
    for st in ('measure', 'aggregate'):
        rec = run012(st, ex, mode='pilot')
        print('>>> %s: %.0f s, sampled peak GPU memory %s MiB (base %s), log %s' % (st, rec['seconds'], rec['vram_peak_sampled_mib'], rec['vram_base_mib'], rec['log']))
    save_results_json('pilot')
    rp = json.load(open(WORK012 + '/results_pilot/results.json'))
    print('pilot results.json: pilot=%s reasons=%s' % (rp['pilot'], rp['pilot_reasons']))
    print('pilot total %.0f s (baseline for 1 train seed at depth 6 + 1 erased unit)' % (time.time() - t0))

In [ ]:
# D2. Pilot timing and a (rough) re-estimate of the main run.
import json
tp = WORK012 + '/timing_pilot.json'
if os.path.exists(tp):
    for r in json.load(open(tp)):
        if r['rc'] == 0 and '(dry)' not in r['label']:
            print('%-22s %7.0f s  peak VRAM (sampled whole GPU) %s MiB  log %s' % (r['label'], r['seconds'], r['vram_peak_sampled_mib'], r['log']))
    print('Main run = 51 depth measurements (18 baseline = 3 seeds x 6 depths, 33 erased = 3 seeds x (5 blocks x 2 depths + block 6 x 1)); the pilot has 2 (1 baseline + 1 erased, both depth 6). The log lines "measured in N s" give the cost of one depth measurement (5 measures).')
else:
    print('no pilot timing yet')
print('Before any measurement, README estimate (추정): about 2 h (range 1.5-3 h) on an L4: 009 intervene 3005.9 s (009 timing_main.json, 21 units) + attentive probes <= about 50 min (004 enc_probe: about 58 s per k/control cell of linear+attentive x 5 seeds, 004 log) + MLP/logistic not measured.')

# E. Main measure (needs `APPROVED_MAIN_012 = True` in cell D0)

In [ ]:
# E1. measure: all units from config (m_int [1..6], train seeds [0,1,2]) = 21 units. Finished units (Drive part files) are skipped; rerun this cell after a disconnect (A and B first).
# Expected GPU time: about 2 h (추정, see README; no measurement). A foreground cell stops with the kernel.
if not APPROVED_MAIN_012:
    print('사람 승인 필요(CLAUDE.md): APPROVED_MAIN_012 = False -> nothing was run.')
else:
    v = T.read_yaml_values(EXP012 + '/config.yaml', ['pilot', 'seed', 'intervention.method', 'intervention.strengths', 'intervention.m_int', 'predictor.train_seeds', 'measure.probe_seeds'])
    print('012 config:', v)
    if v['pilot'] is not False and not ALLOW_MAIN_WITH_PILOT_TRUE:
        raise SystemExit('012 config.yaml has pilot: %s. The human sets pilot: false (README) or ALLOW_MAIN_WITH_PILOT_TRUE=True. Nothing was run.' % v['pilot'])
    assert shutil.disk_usage('/content').free / 1e9 > 20, 'less than 20 GB free on /content (a baseline unit needs up to ~13 GB temporary, 추정)'
    rec = run012('measure', mode='main')
    print('>>> measure done in %.0f s (sampled peak GPU %s MiB), log %s' % (rec['seconds'], rec['vram_peak_sampled_mib'], rec['log']))

# F. Aggregate (CPU; per-block verdicts, results.json)

In [ ]:
# F1. aggregate -> results/results.json (Drive) and <clone>/experiments/012.../results.json (copied). pilot is true in results.json whenever config pilot is true, a CLI override was used, or the results folder differs.
if not APPROVED_MAIN_012:
    print('APPROVED_MAIN_012 = False -> nothing was run (aggregate follows the main measure).')
else:
    rec = run012('aggregate', mode='main')
    save_results_json('main')
    r = json.load(open(WORK012 + '/results/results.json'))
    print({k: r[k] for k in ('pilot', 'pilot_reasons', 'm_int_used', 'm_int_overridden', 'train_seeds', 'method')})
    print({m: b['verdict']['overall']['status'] for m, b in r['blocks'].items()})
    print('missing units:', r['missing_units'])

# G. Copy results out

In [ ]:
# G1. What is on Drive now. Optional zip download (DOWNLOAD_BUNDLE).
DOWNLOAD_BUNDLE = False
import glob
for f in sorted(glob.glob(WORK012 + '/*') + glob.glob(WORK012 + '/results/*') + glob.glob(WORK012 + '/results_pilot/*')):
    print(f, os.path.getsize(f) if os.path.isfile(f) else '(dir)')
for f in ('results/attempts.json', 'results_pilot/attempts.json'):
    if os.path.exists(WORK012 + '/' + f):
        print(WORK012 + '/' + f, open(WORK012 + '/' + f).read().replace('\n', ' '))
if DOWNLOAD_BUNDLE:
    import zipfile
    zp = '/content/exp012_results_bundle.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for ext in ('json', 'log'):
            for f in glob.glob(WORK012 + '/**/*.' + ext, recursive=True):
                z.write(f, os.path.relpath(f, os.path.dirname(WORK012)))
    from google.colab import files
    files.download(zp)

In [ ]:
# G2. Flush Drive writes before leaving.
from google.colab import drive
drive.flush_and_unmount()